## Setting Up a HF transformer model
In this guide, we;ll walk through the steps to set up a Hugging Face (HF) transformer model on Google Colab. Using a GPU is recommended for faster processing and better performance.

1. Select GPU as Hardware Accelerator:

 - Navigate to `Runtime -> Change runtime type` in the Colab menu.

 - Select `GPU` under the Hardware accelerator dropdown menu.

2. Install Required Libraries:


In [ ]:
%pip install --upgrade --quiet transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 19.4 MB/s eta 0:00:00


In [ ]:
pip install --upgrade --quiet langchain-huggingface

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 408.7/408.7 kB 12.3 MB/s eta 0:00:00


In [ ]:
!pip install -qU context-cite fsspec==2024.10.0 langchain-community langchain-openai langchain-core langchain-text-splitters faiss-gpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.4/50.4 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.5/85.5 MB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.3/389.3 kB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 38.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 519.3/519.3 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.3/115.3 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.5/49.5 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 

Next, we'll create a RAG chain using a local txt file (a Wikipedia article about the Transformer architecture) as our "database" to keep things simple.

In [ ]:
import os
import torch as ch
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import CharacterTextSplitter
from langchain_core.runnables import RunnablePassthrough, chain
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_community.llms.huggingface_pipeline import HuggingFacePipeline
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

from context_cite import ContextCiter

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


3. Loading a Hugging Face Transformer Model
We will load a pre-trained transformer model from Hugging Face for causal language modeling.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

checkpoint = "HuggingFaceTB/SmolLM2-1.7B-Instruct"

# for GPU usage or "cpu" for CPU usage
device = "cuda"

# load tokenizer
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

# for multiple GPUs install accelerate
# model = AutoModelForCausalLM.from_pretrained(checkpoint, device_map="auto")`

# for a single device (GPU or CPU)
model = AutoModelForCausalLM.from_pretrained(checkpoint).to(device)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/3.76k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.10M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/792 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.42G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

## Generating Text with a Transformer Model without RAG
We will generate text using the model without utilizing Retrieval-Augmented Generation (RAG). It sets up a prompt about the 2022 FIFA World Cup and uses the model to complete the text.

In [ ]:
from transformers import pipeline

prompt = f"The winner of the 2022 FIFA World Cup was ___ and the MVP was ___"

# Create Messages for the Model
messages = [{"role": "user", "content": f"{prompt}"}]

# Apply Chat Template
# Prepare the input text using the tokenizer's chat template:
input_text = tokenizer.apply_chat_template(messages, tokenize=False)
print("======[input_text]======")
print(input_text)

# Generate Text
inputs = tokenizer.encode(input_text, return_tensors="pt").to(device)
# attention_mask = inputs.ne(tokenizer.pad_token_id).to(device)
outputs = model.generate(inputs, max_new_tokens=50, temperature=0.2, top_p=0.9, do_sample=True)
print("======[outputs]======")
print(tokenizer.decode(outputs[0]))



======[input_text]======
<|im_start|>system
You are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>
<|im_start|>user
The winner of the 2022 FIFA World Cup was ___ and the MVP was ___<|im_end|>

======[outputs]======
<|im_start|>system
You are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>
<|im_start|>user
The winner of the 2022 FIFA World Cup was ___ and the MVP was ___<|im_end|>
<|im_start|>assistant
The winner of the 2022 FIFA World Cup was France and the MVP was Luka Modrić.<|im_end|>


# Use RAG

## Step 1: Embed the Document
In this step, we'll use the sentence-transformers library to embed the document for further processing.

Copy FIFA Knowledge from this [url](https://en.wikipedia.org/wiki/FIFA_World_Cup) and save it into a file `FIFA_cup.txt` and upload to Colab.


In [ ]:
from sentence_transformers import SentenceTransformer, util
from langchain_text_splitters import CharacterTextSplitter

# Load the FIFA cup knowledge
file_path = 'FIFA_cup.txt'
with open(file_path, 'r', encoding='utf-8') as file:
  large_document = file.read()

# Initialize the CharacterTextSplitter
text_splitter = CharacterTextSplitter(chunk_size=500, chunk_overlap=50)

# Split the document into chunks
chunks = text_splitter.split_text(large_document)

for i, text_chunk in enumerate(chunks):
  print(f"=============================== Chunk {i+1} ===============================\n\n{text_chunk}\n\n")

=============================== Chunk 1===============================

The FIFA World Cup is an international association football competition contested by the senior men's national teams of the Fédération Internationale de Football Association (FIFA), the sport's global governing body. The championship has been awarded every four years since 1930, except in 1942 and 1946, when it was not held because of World War II.


=============================== Chunk 2===============================

The World Cup final is the last match of the competition, played by the only two teams remaining in contention, and the result determines which country is declared the world champion. It is a one-off match decided in regulation time. In case of a draw, extra time is used. If scores are then still level, a penalty shoot-out determines the winner,[1] under the rules in force since 1986; prior to that, finals still tied after extra time would have been replayed,[n 1] though this never proved necessary

we will initialize a Sentence Transformer model and use it to embed the text chunks from our document.

In [ ]:
# Initialize the sentence transformer model
embedder = SentenceTransformer('all-MiniLM-L6-v2')

# Encode the text chunks
document_embeddings = embedder.encode(chunks, convert_to_tensor=True)

# Output the embeddings
print(document_embeddings)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.7k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

tensor([[ 0.0387,  0.0784,  0.0051,  ..., -0.0023,  0.0495, -0.0431],
        [ 0.0282,  0.0648,  0.0019,  ...,  0.0376,  0.0145, -0.0086],
        [ 0.0254,  0.0785, -0.0117,  ...,  0.0753, -0.0040,  0.0132],
        ...,
        [-0.0229,  0.0529, -0.0597,  ...,  0.0192,  0.0289,  0.0040],
        [ 0.0058,  0.0203, -0.0101,  ...,  0.0187, -0.0376,  0.0286],
        [ 0.0208,  0.0757, -0.0827,  ...,  0.0127, -0.0619, -0.0127]],
       device='cuda:0')


## Step 2: Retrieve Relevant Information
Now, we'll retrieve the most relevant piece of information from the document based on the user's query:

In [ ]:
query = "Who was the winner of the 2022 FIFA World Cup?"
# Embed the query
query_embedding = embedder.encode(query, convert_to_tensor=True)
# Compute cosine similarity between the query and the document
cosine_scores = util.pytorch_cos_sim(query_embedding, document_embeddings)[0]
# Find the index of the highest score
most_relevant_idx = cosine_scores.argmax().item()
# Get the most relevant sentence
most_relevant_sentence = chunks[most_relevant_idx]

print(most_relevant_sentence)
print(cosine_scores)

Argentina defeated France on penalties in the latest final, staged at Qatar's Lusail Stadium in 2022.
tensor([0.4574, 0.3983, 0.4185, 0.4580, 0.5861, 0.3968, 0.5086],
       device='cuda:0')


##  Step3: Generate Text with Model
Finally, we'll use a generative model to generate text using the retrieved information:

In [ ]:
prompt = f"Based on the information: '{most_relevant_sentence}', the winner of the 2022 FIFA World Cup was ___  and the MVP was ___"
messages = [{"role": "user", "content": f"{prompt}"}]
input_text=tokenizer.apply_chat_template(messages, tokenize=False)
print("======[input_text]======")
print(input_text)

inputs = tokenizer.encode(input_text, return_tensors="pt").to(device)
outputs = model.generate(inputs, max_new_tokens=50, temperature=0.2, top_p=0.9, do_sample=True)
print("======[outputs]======")
print(tokenizer.decode(outputs[0]))

======[input_text]======
<|im_start|>system
You are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>
<|im_start|>user
Based on the information: 'Argentina defeated France on penalties in the latest final, staged at Qatar's Lusail Stadium in 2022.', the winner of the 2022 FIFA World Cup was ___  and the MVP was ___<|im_end|>

======[outputs]======
<|im_start|>system
You are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>
<|im_start|>user
Based on the information: 'Argentina defeated France on penalties in the latest final, staged at Qatar's Lusail Stadium in 2022.', the winner of the 2022 FIFA World Cup was ___  and the MVP was ___<|im_end|>
<|im_start|>assistant
Based on the information provided, the winner of the 2022 FIFA World Cup was Argentina, and the MVP was Lionel Messi.<|im_end|>


## Using Reranking Technique in RAG
we will implement a reranking technique using the `CrossEncoder` from the sentence-transformers library. This approach helps in improving the accuracy of retrieval by scoring and ranking the passages.

In [ ]:
from sentence_transformers.cross_encoder import CrossEncoder
cross_encoder_id = "cross-encoder/ms-marco-MiniLM-L-12-v2"
cross_encoder = CrossEncoder(cross_encoder_id)


config.json:   0%|          | 0.00/791 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/134M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

## Reranking Document Chunks with CrossEncoder
we will use the CrossEncoder model to score and rank the document chunks based on their relevance to a given query.

In [ ]:
# Create Query-Chunk Pairs
query_chunk_pair = []
for chunk in chunks:
  query_chunk_pair.append([query, chunk])

print(query_chunk_pair)

# Predict Scores with CrossEncoder
scores = cross_encoder.predict(query_chunk_pair)
print(scores)

[['Who was the winner of the 2022 FIFA World Cup?', "The FIFA World Cup is an international association football competition contested by the senior men's national teams of the Fédération Internationale de Football Association (FIFA), the sport's global governing body. The championship has been awarded every four years since 1930, except in 1942 and 1946, when it was not held because of World War II."], ['Who was the winner of the 2022 FIFA World Cup?', 'The World Cup final is the last match of the competition, played by the only two teams remaining in contention, and the result determines which country is declared the world champion. It is a one-off match decided in regulation time. In case of a draw, extra time is used. If scores are then still level, a penalty shoot-out determines the winner,[1] under the rules in force since 1986; prior to that, finals still tied after extra time would have been replayed,[n 1] though this never proved necessary. The golden goal rule would have appl

### Reranking Chunks Based on Relevance Scores in descending order
we will rerank the document chunks based on their relevance scores, provided by the `CrossEncoder`, to identify the most relevant chunk of text.

In [ ]:
# Zip Chunks with Scores
reranked_chunk_with_score = list(zip(chunks, scores))

# Sort by score in descending order
reranked_chunk_score_pairs = sorted(reranked_chunk_with_score, key=lambda x: x[1], reverse=True)

# Unzip to get separate ranked lists of texts and scores
reranked_chunk, reranked_scores = zip(*reranked_chunk_score_pairs)
print(reranked_chunk)
print(reranked_scores)
most_relevant_sentence = reranked_chunk[0]

("Argentina defeated France on penalties in the latest final, staged at Qatar's Lusail Stadium in 2022.", "Map of winning countries\nResults by nation\nTeam\tWinners\tRunners-up\tTotal finals\tYears won\tYears runners-up\n Brazil\t5\t2\t7\t1958, 1962, 1970, 1994, 2002\t1950, 1998\n Germany\t4\t4\t8\t1954, 1974, 1990, 2014\t1966, 1982, 1986, 2002\n Italy\t4\t2\t6\t1934, 1938, 1982, 2006\t1970, 1994\n Argentina\t3\t3\t6\t1978, 1986, 2022\t1930, 1990, 2014\n France\t2\t2\t4\t1998, 2018\t2006, 2022\n Uruguay\t2\t0\t2\t1930, 1950\t—\n England\t1\t0\t1\t1966\t—\n Spain\t1\t0\t1\t2010\t—\n Netherlands\t0\t3\t3\t—\t1974, 1978, 2010\n Hungary\t0\t2\t2\t—\t1938, 1954\n Czechoslovakia\t0\t2\t2\t—\t1934, 1962\n Sweden\t0\t1\t1\t—\t1958\n Croatia\t0\t1\t1\t—\t2018\nResults by confederation\nConfederation\tAppearances\tWinners\tRunners-up\nUEFA\t29\t12\t17\nCONMEBOL\t15\t10\t5\nSee also\nList of FIFA Confederations Cup finals\nList of FIFA World Cup final stadiums\nList of FIFA World Cup final goals

### Generate Text with the Top Reranked Chunk
we will generate a response using the most relevant chunk of text identified through the reranking process.

In [ ]:
# Use the top reranked chunk (most_relevant_sentence) to form a prompt for the model.
prompt = f"Based on the information: '{most_relevant_sentence}', the winner of the 2022 FIFA World Cup was ___ and who was the MVP?"

# Create Messages for the Model
messages = [{"role": "user", "content": f"{prompt}"}]
input_text=tokenizer.apply_chat_template(messages, tokenize=False)
print("======[input_text]======")
print(input_text)

# Generate Text
inputs = tokenizer.encode(input_text, return_tensors="pt").to(device)
outputs = model.generate(inputs, max_new_tokens=50, temperature=0.2, top_p=0.9, do_sample=True)
print("======[outputs]======")
print(tokenizer.decode(outputs[0]))

<|im_start|>system
You are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>
<|im_start|>user
Based on the information: 'Argentina defeated France on penalties in the latest final, staged at Qatar's Lusail Stadium in 2022.', the winner of the 2022 FIFA World Cup was ___ and who was the MVP?<|im_end|>

<|im_start|>system
You are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>
<|im_start|>user
Based on the information: 'Argentina defeated France on penalties in the latest final, staged at Qatar's Lusail Stadium in 2022.', the winner of the 2022 FIFA World Cup was ___ and who was the MVP?<|im_end|>
<|im_start|>assistant
Based on the information provided, the winner of the 2022 FIFA World Cup was Argentina, and the MVP (Most Valuable Player) was Lionel Messi.<|im_end|>


## Modify the code and answer these questions by RAG

1. When was AIwarebootcamp hosted?
2. When was the speaker of `RAG engineering`?
3. Which restaurant was chosen for dinner on Day 1?


In [ ]:
prompt = f"Based on the information: '{most_relevant_sentence}', When was AIwarebootcamp hosted? When was the speaker of RAG engineering? Which restaurant was chosen for dinner on Day 1?"

# Create Messages for the Model
messages = [{"role": "user", "content": f"{prompt}"}]
input_text=tokenizer.apply_chat_template(messages, tokenize=False)
print("======[input_text]======")
print(input_text)

# Generate Text
inputs = tokenizer.encode(input_text, return_tensors="pt").to(device)
outputs = model.generate(inputs, max_new_tokens=50, temperature=0.2, top_p=0.9, do_sample=True)
print("======[outputs]======")
print(tokenizer.decode(outputs[0]))

======[input_text]======
<|im_start|>system
You are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>
<|im_start|>user
Based on the information: 'Argentina defeated France on penalties in the latest final, staged at Qatar's Lusail Stadium in 2022.', When was AIwarebootcamp hosted? When was the speaker of RAG engineering? Which restaurant was chosen for dinner on Day 1?<|im_end|>

======[outputs]======
<|im_start|>system
You are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>
<|im_start|>user
Based on the information: 'Argentina defeated France on penalties in the latest final, staged at Qatar's Lusail Stadium in 2022.', When was AIwarebootcamp hosted? When was the speaker of RAG engineering? Which restaurant was chosen for dinner on Day 1?<|im_end|>
<|im_start|>assistant
I'm sorry for the confusion, but as an AI, I don't have real-time access to specific dates or events. However, I can provide general information. AIWarebootcamp was hosted in